In [3]:
import torch
import torch.nn.functional as  F

def ppo_clip_loss(logp_new,logp_old,adv,eps=0.2,reduce = True):
    """
    计算 PPO-Clip 策略损失
    :param logp_new: 新策略 log π_θ(a|s)  [B]
    :param logp_old: 旧策略 log π_θ_old(a|s) [B]
    :param adv:      优势 A(s,a)  [B]  （可为正或负）
    :param eps:      clip 范围 ε
    :param reduce:   是否返回标量
    :return:         PPO-Clip 损失（默认求平均）

    """
    # 概率比 r_t(θ)
    ratio = torch.exp(logp_new - logp_old)

    # 未裁剪项
    surr1 = ratio * adv
    # 裁剪项
    clipped_ratio = torch.clamp(ratio, 1.0 - eps,1.0 + eps)
    surr2 = clipped_ratio * adv
    
    # 取min
    policy_loss = -torch.min(surr1,surr2)

    return policy_loss.mean() if reduce else policy_loss

B = 4 # batch size
logp_old = torch.randn(B).detach() # 旧策略
logp_new = torch.randn(B, requires_grad = True) # 新策略
adv = torch.tensor([+0.8, -0.5, +1.2, -0.3]) # 优势：有正有负
loss = ppo_clip_loss(logp_new,logp_old,adv,eps=0.2)
print("PPO_Clip loss:",loss.item())

# 反向传播
loss.backward()
print("梯度范数",logp_new.grad.norm().item())

PPO_Clip loss: -0.13175201416015625
梯度范数 0.36824798583984375


In [1]:
import numpy as np

# 目标：估计 E_p[f(x)], p(x) = N(5, 1), f(x) = x^2
# 但改用 q(x) = N(0, 1) 采样（较差的选择，仅为演示）

N = 10000
x = np.random.normal(0, 1, N)          # 从 q(x) 采样
p = np.exp(-0.5 * (x - 5)**2) / np.sqrt(2*np.pi)   # p(x)（忽略常数不影响比例）
q = np.exp(-0.5 * x**2) / np.sqrt(2*np.pi)

weights = p / q
f_x = x**2
estimate = np.mean(f_x * weights)
print("IS estimate:", estimate)
print("True value:", 5**2 + 1)  # Var=1, mean=5 → E[x^2]=26

IS estimate: 1958.0881718290982
True value: 26
